# Suite CNF — Conflicting rules

A grid's rules are its children's adjacency specs, in the order they reached it. After each change it settles: each rule kept where it holds, met where one change of the sequence breaks no earlier rule that holds, and ignored, the later of the incompatible rules, where none can, until a change lets it be met, in its turn: mbse-basis [CAP-240: conflicting layout rules ignored and requeued](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-240), for container layout, by the rules of [the diagrams proposal](https://github.com/pitaman71/mbse-basis/blob/main/proposals/DIAGRAMS.md).

In [ ]:
import { Builders, Validators } from "@mbse/schemas/Framework";

import { Container, Logical } from "../src/Framework/index.js";

function assert(condition: unknown, message = "assertion failed"): asserts condition {
  if (!condition) throw new Error(`AssertionError: ${message}`);
}

const store: any = new Builders.Store();
store.transaction(Logical.install);
store.transaction(Container.install);
const L = store.Logical;
const diagrams = store.singleton("Logical.Diagrams");
const problems = (schema: any, root: any): string[] => Validators.Validate(store).Reachable(schema, root);
const text = (value: unknown): string => JSON.stringify(value, (_k, v) => typeof v === "bigint" ? Number(v) : v);

store.transaction(() =>
  L.DiagramDirectory(diagrams, (f: any) => f.diagrams((a: any) => a.insert((e: any) => e.name("overview").diagram((d: any) => d)))));
const overview = diagrams.diagrams({ name: "overview" });

/** A top-level shape of the diagram holding a child of each name in `children`, laid out in a grid. */
function box(name: string, children: string, span: bigint, grows = "rows", cell_rotation = 0): any {
  const spec = (s: any): any => {
    for (const child of children) s.children((k: any) => k.insert((e: any) => e.name(child).child((c: any) => c)));
    return s;
  };
  let shape: any;
  store.transaction(() => {
    L.OfDiagram(overview, (d: any) => d.shapes((s: any) => s.insert((e: any) => e.name(name).shape(spec))));
    shape = overview.shapes({ name });
    Container.layout(shape, span, grows, cell_rotation);
  });
  return shape;
}

/** Adds the adjacency spec from child `a` to child `b`, by `degrees`, and adjoins them: whether it did. */
function beside(shape: any, a: string, b: string, degrees: number): boolean {
  let done = false;
  store.transaction(() => {
    L.OfShape(shape.children({ name: a }), (s: any) => s.adjacent_to((x: any) => x.insert(
      (e: any) => e.to_shape(shape.children({ name: b })).rotation((r: any) => r.degrees(degrees)))));
    done = Container.adjoined(shape.children({ name: a }), shape.children({ name: b }));
  });
  return done;
}

/** Removes the adjacency spec from child `a` to child `b`, and unjoins `b`. */
function apart(shape: any, a: string, b: string): void {
  store.transaction(() => {
    L.OfShape(shape.children({ name: a }), (s: any) => s.adjacent_to((x: any) => x.remove(
      (e: any) => e.to_shape(shape.children({ name: b })))));
    Container.unjoined(shape.children({ name: b }));
  });
}

/** Each cell, in row and column order, as JSON text: its row and column, the names of the children it holds, in
 * order, each with its order, and its axis and extents. */
function cells(shape: any): string {
  const [entry] = [...shape.grid];
  const names = new Map([...shape.children].map((e: any) => [e.child, e.name.get()]));
  const found = [...entry.grid.cells]
    .sort((x: any, y: any) => Number(x.row.get() - y.row.get()) || Number(x.column.get() - y.column.get()))
    .map((e: any) => [[e.row.get(), e.column.get()], [...e.cell.children].map((h: any) => [names.get(h.shape), h.order.get()]),
      e.cell.grows.get(), e.cell.rows.get(), e.cell.columns.get()]);
  return text(found);
}

/** The names of the ignored rules' shapes, and their degrees, in the order the rules reached the grid. */
function queue(shape: any): [string, string, number][] {
  const names = new Map([...shape.children].map((e: any) => [e.child, e.name.get()]));
  return Container.ignored(shape).map(([f, t, d]) => [names.get(f), names.get(t), d]);
}

/** The grid's rules, as data: each one's shapes' names, its degrees and whether it is ignored. */
function rules(shape: any): unknown[] {
  const [entry] = [...shape.grid];
  const names = new Map([...shape.children].map((e: any) => [e.child, e.name.get()]));
  return [...entry.grid.rules].map((e: any) => [names.get(e.from_shape), names.get(e.to_shape), e.degrees.get(),
    e.ignored.get()]);
}

/** Adds a child of `name` to `shape`, and places it. */
function grow(shape: any, name: string): void {
  store.transaction(() => {
    L.OfShape(shape, (s: any) => s.children((k: any) => k.insert((e: any) => e.name(name).child((c: any) => c))));
    Container.added(shape.children({ name }));
  });
}

<a id="CNF-01"></a>
## CNF-01 · A spec that would break an earlier rule that holds is ignored, and the earlier one kept

In [ ]:
const cpu = box("cpu", "abcdef", 3n);
assert(text([beside(cpu, "a", "d", 180), beside(cpu, "a", "b", 270), beside(cpu, "a", "c", 270)]) === "[true,false,false]");
assert(cells(cpu) === "[[[0,0],[[\"a\",0]],null,1,1],[[0,1],[[\"b\",1]],null,1,1],[[0,2],[[\"c\",2]],null,1,1],[[1,0],[[\"d\",3]],null,1,1],[[1,1],[[\"e\",4]],null,1,1],[[1,2],[[\"f\",5]],null,1,1]]");
assert(text(queue(cpu)) === "[[\"a\",\"b\",270],[\"a\",\"c\",270]]");

<a id="CNF-02"></a>
## CNF-02 · A grid's rules are data: each spec, in the order it reached the grid, and whether it is ignored

In [ ]:
assert(text(rules(cpu)) === "[[\"a\",\"d\",180,false],[\"a\",\"b\",270,true],[\"a\",\"c\",270,true]]");

<a id="CNF-03"></a>
## CNF-03 · Removing the earlier spec lifts the conflict: the ignored rules are met, in the order they reached the grid

In [ ]:
apart(cpu, "a", "d");
assert(cells(cpu) === "[[[0,0],[[\"b\",1],[\"c\",2]],\"rows\",2,1],[[0,1],[[\"a\",0]],null,1,1],[[0,2],[[\"d\",3]],null,1,1],[[1,0],[[\"e\",4]],null,1,1],[[1,1],[[\"f\",5]],null,1,1]]");
assert(text(queue(cpu)) === "[]");
assert(text(rules(cpu)) === "[[\"a\",\"b\",270,false],[\"a\",\"c\",270,false]]");

<a id="CNF-04"></a>
## CNF-04 · Removing a child lifts a conflict its rules made

In [ ]:
const gpu = box("gpu", "abcd", 3n);
assert(text([beside(gpu, "a", "d", 180), beside(gpu, "a", "b", 270), queue(gpu)]) === "[true,false,[[\"a\",\"b\",270]]]");
store.transaction(() => {
  L.OfShape(gpu, (s: any) => s.children((k: any) => k.remove((e: any) => e.name("d"))));
  Container.removed(gpu);
});
assert(cells(gpu) === "[[[0,0],[[\"b\",1]],null,1,1],[[0,1],[[\"a\",0]],null,1,1],[[0,2],[[\"c\",2]],null,1,1]]");
assert(text(queue(gpu)) === "[]");

<a id="CNF-05"></a>
## CNF-05 · Adding children lifts a conflict with the grid's extents

In [ ]:
const mmu = box("mmu", "ab", 3n);
assert(text([beside(mmu, "a", "b", 180), queue(mmu)]) === "[false,[[\"a\",\"b\",180]]]");
grow(mmu, "c");
assert(cells(mmu) === "[[[0,0],[[\"a\",0]],null,1,1],[[0,1],[[\"b\",1]],null,1,1],[[0,2],[[\"c\",2]],null,1,1]]");
assert(text(queue(mmu)) === "[[\"a\",\"b\",180]]");
grow(mmu, "d");
assert(cells(mmu) === "[[[0,0],[[\"a\",0]],null,1,1],[[0,1],[[\"c\",2]],null,1,1],[[0,2],[[\"d\",3]],null,1,1],[[1,0],[[\"b\",1]],null,1,1]]");
assert(text(queue(mmu)) === "[]");

<a id="CNF-06"></a>
## CNF-06 · A child reverting out of a shared cell leaves the rules that hold in place

In [ ]:
const io = box("io", "pqrst", 3n);
assert(text([beside(io, "p", "r", 180), beside(io, "p", "t", 180)]) === "[true,true]");
assert(cells(io) === "[[[0,0],[[\"p\",0]],null,1,1],[[0,1],[[\"q\",1]],null,1,1],[[0,2],[[\"s\",3]],null,1,1],[[1,0],[[\"r\",2],[\"t\",4]],\"rows\",2,1]]");
apart(io, "p", "r");
assert(cells(io) === "[[[0,0],[[\"p\",0]],null,1,1],[[0,1],[[\"q\",1]],null,1,1],[[0,2],[[\"r\",2]],null,1,1],[[1,0],[[\"t\",4]],null,1,1],[[1,1],[[\"s\",3]],null,1,1]]");
assert(text([queue(io), [cpu, gpu, mmu, io].map((s) => problems(L.OfShape, s))]) === "[[],[[],[],[],[]]]");

<a id="CNF-07"></a>
## CNF-07 · Removing an ignored spec moves nothing

In [ ]:
const dsp = box("dsp", "abcd", 3n);
assert(text([beside(dsp, "a", "d", 180), beside(dsp, "a", "b", 270)]) === "[true,false]");
assert(cells(dsp) === "[[[0,0],[[\"a\",0]],null,1,1],[[0,1],[[\"b\",1]],null,1,1],[[0,2],[[\"c\",2]],null,1,1],[[1,0],[[\"d\",3]],null,1,1]]");
apart(dsp, "a", "b");
assert(cells(dsp) === "[[[0,0],[[\"a\",0]],null,1,1],[[0,1],[[\"b\",1]],null,1,1],[[0,2],[[\"c\",2]],null,1,1],[[1,0],[[\"d\",3]],null,1,1]]");
assert(text([queue(dsp), rules(dsp)]) === "[[],[[\"a\",\"d\",180,false]]]");